# 예제 V · Token Merging (ToMe) (수강생용)

같은 허스키 이미지에서 ToMe의 병합 설정 `r`만 변경합니다. ToMe를 적용하지 않은 것과 같은 조건인 `r=0`을 기준으로, 마지막 블록의 Token 수와 Top-5 클래스 목록을 비교합니다.

## 1. 라이브러리, 모델, 예제 이미지 준비

셀 1~3을 처음에 한 번 실행합니다. 정답용 예제에 쓰인 ViT-Base 가중치는 첫 실행에서 내려받습니다.

In [ ]:
import importlib.metadata
import os
from pathlib import Path
import subprocess
import sys
import time

import torch
import matplotlib.pyplot as plt
from PIL import Image

torch.set_num_threads(min(4, os.cpu_count() or 1))
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ASSET_ROOT = Path(os.environ.get("AX_IDEC_CACHE", "./idec_student_cache"))
ASSET_ROOT.mkdir(parents=True, exist_ok=True)
try:
    timm_version = importlib.metadata.version("timm")
except importlib.metadata.PackageNotFoundError:
    timm_version = None
# 공식 ToMe 구현과 호환되는 timm 버전을 준비한다.
if timm_version != "0.9.10":
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "timm==0.9.10"])
print("실행 장치:", DEVICE)

In [ ]:
from urllib.request import urlretrieve
from zipfile import ZipFile
import sys

# 체크포인트와 소스를 캐시에 보관한다. 다운로드가 완료되면 임시 파일을 정상 파일 이름으로 바꾼다.
def fetch(url, filename):
    destination = ASSET_ROOT / filename
    if not destination.is_file():
        pending = destination.with_suffix(destination.suffix + ".download")
        try:
            urlretrieve(url, pending)
            pending.replace(destination)
        finally:
            pending.unlink(missing_ok=True)
    return destination

# 고정한 커밋의 ZIP을 받고, 각 항목의 경로를 검사한 뒤 추출한다.
def fetch_repository(owner, repository, revision, filename):
    archive = fetch(f"https://codeload.github.com/{owner}/{repository}/zip/{revision}", filename)
    root = ASSET_ROOT / f"{repository}-{revision}"
    if not root.exists():
        with ZipFile(archive) as source_zip:
            for entry in source_zip.infolist():
                destination = (ASSET_ROOT / entry.filename).resolve()
                if not destination.is_relative_to(root.resolve()):
                    raise ValueError(f"Unexpected archive path: {entry.filename}")
            source_zip.extractall(ASSET_ROOT)
    return root.resolve()

In [ ]:
import json
import timm
from torchvision import transforms
from torchvision.transforms.functional import InterpolationMode

# 고정한 ToMe 소스와 예제 Husky 이미지를 캐시에서 가져온다.
revision = "af95e4b1befa172dadccd8c81e223b10090f9579"
tome_root = fetch_repository("facebookresearch", "ToMe", revision, "tome.zip")
sys.path.insert(0, str(tome_root))
import tome

# 정답용과 같은 사전 학습 ViT-Base를 불러와 ToMe를 적용한다.
model = timm.create_model("vit_base_patch16_224", pretrained=True)
model.to(DEVICE).eval()
tome.patch.timm(model)

image = Image.open(tome_root / "examples" / "images" / "husky.png").convert("RGB")
# 정답용 예제와 동일한 크기 조정과 정규화를 적용한다.
input_size = model.default_cfg["input_size"][1]
transform = transforms.Compose([
    transforms.Resize(int((256 / 224) * input_size), interpolation=InterpolationMode.BICUBIC),
    transforms.CenterCrop(input_size),
    transforms.ToTensor(),
    transforms.Normalize(model.default_cfg["mean"], model.default_cfg["std"]),
])
image_tensor = transform(image).unsqueeze(0).to(DEVICE)
labels_file = fetch(
    "https://storage.googleapis.com/download.tensorflow.org/data/imagenet_class_index.json",
    "imagenet_class_index.json"
)
class_names = json.loads(labels_file.read_text(encoding="utf-8"))

plt.figure(figsize=(3.2, 3.2))
plt.imshow(image)
plt.axis("off")
plt.show()

## 2. 병합 설정 r과 Top-5 예측

**셀 4**의 `MERGE_TOKENS`를 0, 8, 16으로 각각 바꿔 이 셀만 다시 실행합니다. 같은 이미지에서 기준 r=0과 선택값의 Top-5 클래스·순서 및 마지막 Token 수를 비교합니다.

In [ ]:
# @title Token 병합 설정 선택
MERGE_TOKENS = 8 # @param {type:"slider", min:0, max:16, step:1}

# 지정한 r로 추론하고, Top-5 확률 및 마지막 Block의 실제 Token 수를 반환한다.
@torch.inference_mode()
def predict(merge_tokens):
    model.r = merge_tokens
    lengths = []
    def count_tokens(_module, _args, output):
        lengths.append(output.shape[1])
    hook = model.blocks[-1].register_forward_hook(count_tokens)
    try:
        probabilities = model(image_tensor).softmax(1)
    finally:
        hook.remove()
    values, indices = probabilities[0].topk(5)
    return [(class_names[str(i)][1], p) for p, i in zip(values.tolist(), indices.tolist())], lengths[-1]


# 같은 입력으로 기본 조건과 선택한 조건을 비교한다.
conditions = list(dict.fromkeys((0, MERGE_TOKENS)))
results = {r: predict(r) for r in conditions}
for r, (top5, tokens) in results.items():
    print(f"r={r} | 마지막 블록 Token 수: {tokens} (클래스 Token 포함)")
    for rank, (name, probability) in enumerate(top5, 1):
        print(f"  {rank}. {name:<25} {probability:.2%}")

if len(results) == 2:
    before_names = [name for name, _ in results[0][0]]
    after_names = [name for name, _ in results[MERGE_TOKENS][0]]
    changed = [str(i) for i, (a, b) in enumerate(zip(before_names, after_names), 1) if a != b]
    print("기준과 다른 순위:", ", ".join(changed) if changed else "없음")

values = [results[r][1] for r in conditions]
fig, ax = plt.subplots(figsize=(5.5, 3.2), constrained_layout=True)
bars = ax.bar([f"r={r}" for r in conditions], values, color=["#64748b", "#167d9a"][:len(values)])
ax.bar_label(bars, padding=3)
ax.set_ylim(0, max(values) * 1.18)
ax.set_ylabel("Tokens after the last block")
plt.show()

## 추가로 바꿔볼 설정 · 선택 실습

- 같은 이미지에서 `MERGE_TOKENS`를 4 또는 12로 바꿉니다. 최종 Token 수와 Top-5 순서를 기준값 r=0과 비교합니다.
- 병합 설정을 고정하고 셀 3의 입력 이미지 파일만 다른 이미지로 바꿉니다. 새 이미지에서 r=0과 선택값을 비교합니다.